# 01 — Stage 1 GRPO (Math), LoRA, group 8

**Requires notebook 00 to have passed every gate and been committed.** LoRA lr=2e-5 is UNVALIDATED (plan §1.1/§8 item 3). Group size 8 and reward `exact_plus_boxed_format_0.1` match the WIN4070 v9 recipe (plan §1.4) — this is the first time group 8 will run to completion.

In [ ]:
import subprocess, sys, os, json
from pathlib import Path

# Private repo: reads a token from Colab's own Secrets store (key icon,
# left sidebar) — add one named GITHUB_TOKEN (a GitHub PAT with repo read
# access) before running this cell. The token is never written to this
# notebook's source and this cell never prints it.
from google.colab import userdata
try:
    _token = userdata.get('GITHUB_TOKEN')
except Exception:
    _token = None
if not _token:
    raise RuntimeError(
        'Add a GITHUB_TOKEN secret (key icon, left sidebar) with repo read '
        'access to WYR186/RLVR, enable notebook access for it, then re-run.')
REPO_URL = f'https://{_token}@github.com/WYR186/RLVR.git'
REPO_DIR = '/content/RLVR'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
# strip the token back out of the stored remote URL immediately — no need
# to leave it sitting in .git/config for the rest of the session
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
                'https://github.com/WYR186/RLVR.git'], check=True)
del _token, REPO_URL  # don't leave the token bound in the notebook's live namespace

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('merge note:', CONFIG['merge_note'])

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
splits = json.load(open(DATA_DIR / 'exp2_colab_splits.json'))
math_rows, _ = guru_data.load_all_records(
    MODEL_ID, MODEL_REVISION, DATASET_REVISION,
    stage_a_prompt_suffix=splits.get('stage_a_prompt_suffix'))
stage_a_rows = guru_data.dataset_rows_for(
    'a', None, splits, MODEL_ID, MODEL_REVISION, DATASET_REVISION)
stage_a_train = guru_data.to_hf_dataset(stage_a_rows)
print('stage-A train rows:', len(stage_a_train))

In [ ]:
RUN_DIR = f'{EXP2_DIR}/../eaaj-pilot/outputs/exp2_colab_guru_math7b_group8_REPLACE_WITH_HASH'
# Replace REPLACE_WITH_HASH with a short content hash of this config once
# decided (matches eaaj-pilot's run-dir convention, src/repro.py:config_hash).
sa = CONFIG['stage_a']
summary = pipeline.run_stage_a_grpo(
    MODEL_ID, CONFIG['peft'], stage_a_train, f'{RUN_DIR}/stage_a',
    checkpoint_steps=sa['checkpoint_steps'], max_steps=sa['max_steps'],
    reward_mode=sa['reward_mode'],
    learning_rate=sa['learning_rate'], per_device_batch=sa['per_device_train_batch_size'],
    grad_accum=sa['gradient_accumulation_steps'], num_generations=sa['num_generations'],
    beta=sa['beta'], temperature=sa['temperature'], top_p=sa['top_p'],
    max_completion_length=sa['max_completion_length'],
    revision=MODEL_REVISION, seed=CONFIG['seed'], eval_every=sa['eval_every'])
print(summary)

## Commit reminder

Commit the run directory (dashboard.jsonl, update_sentinel.jsonl, summary.json, ckpt-0/100/200 adapters), prefix `exp2-colab:`. Log wall time + compute-unit cost in `eaaj-pilot/compute_log.md`. If Phase-0's measured s/update implies the de-scope fallback is needed, apply `de_scope_fallback_max_steps` / `de_scope_fallback_checkpoint_steps` from the config and re-run this notebook — do not silently mix a partial 200-update attempt with a restarted 100-update one.